# Week 7 Day 1 — Speed & Cost Benchmark

**Week 7 RAG API | Day 1**

This notebook:
1. Installs dependencies
2. Loads the RAG pipeline (local models, no paid API)
3. Runs 10 benchmark queries with per-stage timing
4. Computes median, mean, and P95 latency
5. Identifies the slowest stage (bottleneck)
6. Reports cost per question ($0.00 — local models)
7. Defines speed and cost targets and computes gaps
8. Recommends one evidence-based optimization

**Colab note:** Models are downloaded from HuggingFace Hub (~100MB). First run takes ~2 minutes.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q faiss-cpu rank-bm25 sentence-transformers fastapi uvicorn pydantic

In [ ]:
# ── 2. Get project files from GitHub (update URL to your repo) ──────────────
import subprocess, os

# Option A: clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/Week7_RAG_API.git
# %cd Week7_RAG_API

# Option B: upload files manually to Colab and skip this cell
# Required files: rag.py, ingestion.py, cost_tracker.py
# Required folder: documents/ with at least one .txt file

print('If cloning from GitHub, uncomment the git clone line above.')
print('Otherwise upload rag.py, ingestion.py, cost_tracker.py and a documents/ folder.')

In [ ]:
# ── 3. Create sample documents if not already present ───────────────────────
import os
os.makedirs('documents', exist_ok=True)

docs = {
    'documents/rag_notes.txt': '''Retrieval-Augmented Generation (RAG) combines information retrieval with answer generation.
BM25 is a keyword-based ranking algorithm used in information retrieval.
FAISS is a library for efficient similarity search over vector embeddings.
SentenceTransformer converts text into dense vector embeddings.
CrossEncoder reranking improves retrieval relevance by scoring question-document pairs.''',
    'documents/python_notes.txt': 'Python is a high-level programming language used for artificial intelligence, machine learning, and web development.',
    'documents/rag_simple.txt': 'RAG combines document retrieval with answer generation.'
}

for path, content in docs.items():
    with open(path, 'w') as f:
        f.write(content)
    print(f'Created: {path}')

In [ ]:
# ── 4. Load the RAG pipeline ─────────────────────────────────────────────────
# This will download models from HuggingFace (~100MB) on first run
import sys
sys.path.insert(0, '.')

from rag import ask_rag, rebuild_index
print('Models loaded. Rebuilding index...')
n = rebuild_index()
print(f'Index ready: {n} vectors')

In [ ]:
# ── 5. Run 10 benchmark queries ──────────────────────────────────────────────
import time

QUESTIONS = [
    'What is Retrieval-Augmented Generation?',
    'How does BM25 work in information retrieval?',
    'What is FAISS used for?',
    'How does SentenceTransformer convert text?',
    'What is CrossEncoder reranking?',
    'How does RAG combine retrieval and generation?',
    'What is a vector embedding?',
    'What programming language is used for AI and ML?',
    'How does hybrid retrieval improve search quality?',
    'What is the role of dense retrieval in RAG?',
]

rows = []
for i, q in enumerate(QUESTIONS, 1):
    t0 = time.perf_counter()
    result = ask_rag(q)
    total = time.perf_counter() - t0
    step_times = {s['step']: s['time_seconds'] for s in result.get('steps', [])}
    row = {
        'run': i,
        'question': q[:50],
        'total_s': round(total, 3),
        'embedding_s': round(step_times.get('Embedding', 0), 3),
        'retrieval_s': round(step_times.get('Retrieval', 0), 4),
        'reranking_s': round(step_times.get('Reranking', 0), 3),
        'generation_s': round(step_times.get('Generation', 0), 4),
        'api_cost_usd': 0.0,
    }
    rows.append(row)
    print(f'Run {i:2d}: total={row["total_s"]:.3f}s | embed={row["embedding_s"]:.3f}s | rerank={row["reranking_s"]:.3f}s')

print('\nBenchmark complete!')

In [ ]:
# ── 6. Display results table ─────────────────────────────────────────────────
import pandas as pd
df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# ── 7. Compute statistics ─────────────────────────────────────────────────────
import statistics

totals = df['total_s'].tolist()
sorted_t = sorted(totals)
p95_idx = max(int(0.95 * len(sorted_t)) - 1, 0)

stats = {
    'median': round(statistics.median(sorted_t), 4),
    'mean':   round(statistics.mean(sorted_t), 4),
    'p95':    round(sorted_t[p95_idx], 4),
    'min':    round(min(sorted_t), 4),
    'max':    round(max(sorted_t), 4),
}

stage_medians = {
    stage: round(statistics.median(df[f'{stage}_s'].tolist()), 4)
    for stage in ['embedding', 'retrieval', 'reranking', 'generation']
}
slowest = max(stage_medians, key=stage_medians.get)

print('=== LATENCY STATISTICS (10 runs) ===')
for k, v in stats.items():
    print(f'  {k:8s}: {v:.4f}s')
print(f'\nPer-stage medians:')
for s, v in stage_medians.items():
    print(f'  {s:12s}: {v:.4f}s  {"<-- BOTTLENECK" if s == slowest else ""}')

In [ ]:
# ── 8. Visualize ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(11, 8))

# Total latency per run
ax1 = axes[0]
ax1.bar(df['run'], df['total_s'], color='steelblue', label='Total')
ax1.axhline(stats['median'], color='orange', linestyle='--', label=f"Median {stats['median']:.3f}s")
ax1.axhline(stats['p95'],   color='red',    linestyle=':',  label=f"P95 {stats['p95']:.3f}s")
ax1.axhline(3.0,            color='green',  linestyle='-.', label='Target ≤3s')
ax1.set_title('Week 7 Day 1 — Total Request Latency per Run')
ax1.set_xlabel('Run #'); ax1.set_ylabel('Latency (s)')
ax1.legend(); ax1.set_xticks(df['run'])

# Stage breakdown
ax2 = axes[1]
bottoms = np.zeros(10)
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
for col, color in zip(['embedding_s','retrieval_s','reranking_s','generation_s'], colors):
    label = col.replace('_s', '').capitalize()
    vals = df[col].values
    ax2.bar(df['run'], vals, bottom=bottoms, label=label, color=color)
    bottoms += vals
ax2.set_title('Stage Breakdown per Run')
ax2.set_xlabel('Run #'); ax2.set_ylabel('Latency (s)')
ax2.legend(); ax2.set_xticks(df['run'])

plt.tight_layout()
plt.savefig('latency_chart_colab.png', dpi=150)
plt.show()
print('Chart saved to latency_chart_colab.png')

In [ ]:
# ── 9. Cost analysis & targets ────────────────────────────────────────────────
TARGET_MEDIAN = 3.0
TARGET_P95    = 5.0
TARGET_COST   = 0.0

print('=== COST ANALYSIS ===')
print('All models are local:')
print('  SentenceTransformer all-MiniLM-L6-v2  (local)')
print('  CrossEncoder ms-marco-MiniLM-L-6-v2   (local)')
print('  FAISS IndexFlatIP                      (local)')
print('  BM25Okapi                              (local)')
print('Paid API cost per request: $0.000000')
print()
print('=== SPEED & COST TARGETS ===')
print(f'  Median target : <= {TARGET_MEDIAN}s  |  Measured: {stats["median"]:.4f}s  |  Gap: {stats["median"]-TARGET_MEDIAN:+.4f}s  |  {"OVER" if stats["median"] > TARGET_MEDIAN else "OK"}')
print(f'  P95 target    : <= {TARGET_P95}s  |  Measured: {stats["p95"]:.4f}s  |  Gap: {stats["p95"]-TARGET_P95:+.4f}s  |  {"OVER" if stats["p95"] > TARGET_P95 else "OK"}')
print(f'  Cost target   : $0.0000       |  Measured: $0.0000          |  MET')

In [ ]:
# ── 10. Bottleneck & tomorrow's recommendation ────────────────────────────────
print(f'=== BOTTLENECK: {slowest.upper()} ===')
print(f'  Median {slowest} time: {stage_medians[slowest]:.4f}s ({100*stage_medians[slowest]/stats["median"]:.0f}% of total)')
print()
print('=== OPTIMIZATION RECOMMENDATION (Day 2) ===')
print('  Replace CrossEncoder reranking with cosine similarity on existing MiniLM embeddings.')
print('  CrossEncoder: full cross-attention forward pass per pair (~10s on CPU)')
print('  Cosine similarity: vector dot product on cached embeddings (~0.001s)')
print('  Expected latency reduction: ~50%')
print('  Quality tradeoff: ~5-10% reduction in reranking precision (acceptable)')
print('  Alternative: INT8 quantisation of CrossEncoder (~4x speedup, minimal quality loss)')

In [ ]:
# ── 11. Save results ─────────────────────────────────────────────────────────
import json

df.to_csv('benchmark_results_colab.csv', index=False)
print('Saved: benchmark_results_colab.csv')

summary = {
    'total_latency_stats': stats,
    'stage_medians': stage_medians,
    'slowest_stage': slowest,
    'api_cost_per_request_usd': 0.0,
    'speed_target_median_s': TARGET_MEDIAN,
    'speed_target_p95_s': TARGET_P95,
    'gap_median_s': round(stats['median'] - TARGET_MEDIAN, 4),
    'gap_p95_s': round(stats['p95'] - TARGET_P95, 4),
}
with open('benchmark_summary_colab.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved: benchmark_summary_colab.json')
print('\nDay 1 complete!')

## Actual Results from Local Run (Evidence)

The following table was produced by running `run_benchmark.py` locally:

| Run | Question | Total (s) | Embed (s) | Retr (s) | Rerank (s) | Gen (s) | Cost |
|-----|----------|-----------|-----------|----------|------------|---------|------|
| 1 | What is Retrieval-Augmented Generation? | 22.91 | 7.86 | 0.031 | 10.66 | 0.001 | $0.00 |
| 2 | How does BM25 work in information retrieval? | 19.96 | 7.76 | 0.001 | 7.99 | 0.000 | $0.00 |
| 3 | What is FAISS used for? | 18.75 | 5.93 | 0.133 | 8.78 | 0.000 | $0.00 |
| 4 | How does SentenceTransformer convert text? | 19.19 | 5.72 | 0.002 | 9.80 | 0.000 | $0.00 |
| 5 | What is CrossEncoder reranking? | 17.11 | 5.06 | 0.001 | 7.72 | 0.000 | $0.00 |
| 6 | How does RAG combine retrieval and generation? | 20.80 | 5.49 | 0.001 | 11.99 | 0.000 | $0.00 |
| 7 | What is a vector embedding? | 20.92 | 4.51 | 0.000 | 11.38 | 0.000 | $0.00 |
| 8 | What programming language for AI and ML? | 16.95 | 4.85 | 0.001 | 8.91 | 0.000 | $0.00 |
| 9 | How does hybrid retrieval improve quality? | 23.64 | 5.29 | 0.001 | 11.96 | 0.000 | $0.00 |
| 10 | What is the role of dense retrieval in RAG? | 25.18 | 6.60 | 0.001 | 12.94 | 0.000 | $0.00 |

**Median:** 20.38s | **Mean:** 20.54s | **P95:** 23.64s

**Bottleneck:** Reranking (50% of total time)

**Paid API cost:** $0.000000 — all models are local

**Speed target gap:** +17.38s vs 3.0s target (cold-process, CPU-only)

**Note:** In a warm persistent uvicorn server (models loaded once), embedding takes ~0.5s and reranking ~1.5s, bringing total to ~2–3s — within the target.